DATA FRAME "df_lenders"

In [51]:
import gc
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# 1. Define explicit file paths from the scratchpad directory
path_lenders = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\lenders.csv"

# Load the source datasets into memory
df_lenders = pd.read_csv(path_lenders)

In [52]:
print(df_lenders.shape)
print(df_lenders.columns)
print(df_lenders.info())
df_lenders.head(50)

(2349174, 11)
Index(['permanent_name', 'display_name', 'city', 'state', 'country_code',
       'member_since', 'occupation', 'loan_because', 'loan_purchase_num',
       'invited_by', 'num_invited'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 2349174 entries, 0 to 2349173
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   permanent_name     str    
 1   display_name       str    
 2   city               str    
 3   state              str    
 4   country_code       str    
 5   member_since       int64  
 6   occupation         str    
 7   loan_because       str    
 8   loan_purchase_num  float64
 9   invited_by         str    
 10  num_invited        int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 275.4 MB
None


,permanent_name,display_name,city,state,country_code,member_since,occupation,loan_because,loan_purchase_num,invited_by,num_invited
0,qian3013,Qian,NaN,NaN,NaN,1461300457,NaN,NaN,1.0,NaN,0
1,reena6733,Reena,NaN,NaN,NaN,1461300634,NaN,NaN,9.0,NaN,0
2,mai5982,Mai,NaN,NaN,NaN,1461300853,NaN,NaN,NaN,NaN,0
3,andrew86079135,Andrew,NaN,NaN,NaN,1461301091,NaN,NaN,5.0,Peter Tan,0
4,nguyen6962,Nguyen,NaN,NaN,NaN,1461301154,NaN,NaN,NaN,NaN,0
5,sirinapa6764,Sirinapa,NaN,NaN,NaN,1461301496,NaN,NaN,1.0,NaN,0
6,rene7585,Rene,NaN,NaN,NaN,1461301636,NaN,NaN,2.0,NaN,0
7,harald2826,Harald,NaN,NaN,NaN,1461301670,NaN,NaN,2.0,NaN,0
8,mehdi2903,Mehdi,NaN,NaN,NaN,1461301756,NaN,NaN,NaN,NaN,0
9,youchan8125,Youchan,NaN,NaN,NaN,1461301941,NaN,NaN,1.0,NaN,0


In [53]:
# Gather all structural metrics
missing_count = df_lenders.isnull().sum()
missing_percent = (df_lenders.isnull().sum() / len(df_lenders)) * 100
column_types = df_lenders.dtypes

# Combine them into a single DataFrame
full_report_lenders = pd.DataFrame({
    "Data Type": column_types,
    "Missing Count": missing_count,
    "Missing %": missing_percent
})

# Note: We do not sort by "Missing %" here so that the columns 
# stay in their original order (0 to 102), making slicing accurate.

# Columns 1 to 25
print(full_report_lenders.iloc[0:25])

# Strip hidden spaces from categorical string columns
cols_lenders = [
    "permanent_name",
    "display_name",
    "city",
    "state",
    "country_code",
    "member_since",
    "occupation",
    "loan_because",
    "loan_purchase_num",
    "invited_by",
    "num_invited"
]

                  Data Type  Missing Count  Missing %
permanent_name          str              0   0.000000
display_name            str           2770   0.117914
city                    str        1619307  68.930909
state                   str        1713527  72.941681
country_code            str        1458635  62.091399
member_since          int64              0   0.000000
occupation              str        1844598  78.521131
loan_because            str        2174852  92.579434
loan_purchase_num   float64         894281  38.067891
invited_by              str        1852349  78.851077
num_invited           int64              0   0.000000


In [54]:
# View all unique values for the num_invited column
print(df_lenders["num_invited"].unique())

[    0     1     2     4     3     5    11     6    20    13    17     7
    19    10    21     9     8    35    15    16   303    12    45   140
    30    14    18    29    37    43    31    63    53    41    54    49
    25   309    33    50    36    94    42    51    23    34    52    24
    76    27    99    28    46    32    38    70   245    22    61    26
   166    48    95    64   103    55    67    84    56    57   152    44
   191    60    40   100    72    62    88    75   111    69    65    66
   308    78   121   378   164   137   110   129    39    89   183   128
    71    58    93    59  1859    74 24854   356    83   112   144   156
    79   516   236   295   362   266    68   209   259   143  1553   252
  2611   154   238   260   204   234   105   106   124   632   185  1013
   277   324   142  1355   141    86   107   108   138   719   192    85
   127   119   139   417   214   768    73   198   342   169    91  5083
   285   421   130   337   165   216    77   168   

In [55]:
# 1. Initialize a clean copy of the raw data
df_lenders_clean = df_lenders.copy()

# 2. Clean whitespace spaces out of all validated text string columns
string_cols_lenders = ['permanent_name', 'display_name', 'city', 'state', 'country_code', 'occupation', 'loan_because', 'invited_by']
for col in string_cols_lenders:
    if col in df_lenders_clean.columns:
        df_lenders_clean[col] = df_lenders_clean[col].astype(str).str.strip()

# 3. Convert Unix epoch integer timestamps into true datetime format
df_lenders_clean['member_since'] = pd.to_datetime(df_lenders_clean['member_since'], unit='s', errors='coerce')

# 4. Safely deduplicate rows using the true primary key
df_lenders_clean = df_lenders_clean.drop_duplicates(subset=['permanent_name'])

# 5. Verify the structural shape and real data types
print("=== CORRECTED LENDERS CLEANING VERIFICATION ===")
print(f"Cleaned Shape: {df_lenders_clean.shape}")
print("\nTrue Data Types for Key Fields:")
print(df_lenders_clean.dtypes)

=== CORRECTED LENDERS CLEANING VERIFICATION ===
Cleaned Shape: (2349174, 11)

True Data Types for Key Fields:
permanent_name                 str
display_name                   str
city                           str
state                          str
country_code                   str
member_since         datetime64[s]
occupation                     str
loan_because                   str
loan_purchase_num          float64
invited_by                     str
num_invited                  int64
dtype: object


In [56]:
df_lenders_clean.head(50)

,permanent_name,display_name,city,state,country_code,member_since,occupation,loan_because,loan_purchase_num,invited_by,num_invited
0,qian3013,Qian,NaN,NaN,NaN,2016-04-22 04:47:37,NaN,NaN,1.0,NaN,0
1,reena6733,Reena,NaN,NaN,NaN,2016-04-22 04:50:34,NaN,NaN,9.0,NaN,0
2,mai5982,Mai,NaN,NaN,NaN,2016-04-22 04:54:13,NaN,NaN,NaN,NaN,0
3,andrew86079135,Andrew,NaN,NaN,NaN,2016-04-22 04:58:11,NaN,NaN,5.0,Peter Tan,0
4,nguyen6962,Nguyen,NaN,NaN,NaN,2016-04-22 04:59:14,NaN,NaN,NaN,NaN,0
5,sirinapa6764,Sirinapa,NaN,NaN,NaN,2016-04-22 05:04:56,NaN,NaN,1.0,NaN,0
6,rene7585,Rene,NaN,NaN,NaN,2016-04-22 05:07:16,NaN,NaN,2.0,NaN,0
7,harald2826,Harald,NaN,NaN,NaN,2016-04-22 05:07:50,NaN,NaN,2.0,NaN,0
8,mehdi2903,Mehdi,NaN,NaN,NaN,2016-04-22 05:09:16,NaN,NaN,NaN,NaN,0
9,youchan8125,Youchan,NaN,NaN,NaN,2016-04-22 05:12:21,NaN,NaN,1.0,NaN,0


In [57]:
df_lenders_clean.tail(50)

,permanent_name,display_name,city,state,country_code,member_since,occupation,loan_because,loan_purchase_num,invited_by,num_invited
2349124,william69447495,William,Benton,AR,US,2012-07-12 13:20:26,NaN,NaN,NaN,NaN,0
2349125,benno6089,Benno,Washington,District of Columbia,US,2012-07-12 13:47:40,NaN,NaN,4.0,E. Chloe',0
2349126,frank6457,Frank,NaN,NaN,NaN,2012-07-12 13:58:27,NaN,NaN,NaN,NaN,0
2349127,scott2350,Scott,NaN,NaN,NaN,2012-07-12 14:03:45,NaN,NaN,13.0,NaN,0
2349128,kate3496,Kate,Brisbane,Qld,AU,2012-07-12 11:29:32,NaN,NaN,1.0,Brendan,0
2349129,jonathan8921,Jonathan,NaN,NaN,NaN,2012-07-12 12:11:37,NaN,NaN,10.0,NaN,0
2349130,susan16382795,susan,NaN,NaN,NaN,2012-07-12 12:25:47,NaN,NaN,NaN,NaN,0
2349131,andrew3939,Andrew,NaN,NaN,NaN,2012-07-12 12:33:22,NaN,NaN,NaN,NaN,0
2349132,buyarimidex,buy,Richmond,Richmond,CA,2012-07-12 13:15:41,Pharmacy,Buy Arimidex at a discount from Canada Drug Ph...,NaN,NaN,0
2349133,eltrina8605,Eltrina,NaN,NaN,NaN,2012-07-12 13:18:26,NaN,NaN,NaN,NaN,0


In [58]:
# Standardize string missing placeholders to a uniform token
string_cols = ["city", "state", "country_code", "occupation", "loan_because", "invited_by", "display_name"]
for col in string_cols:
    if col in df_lenders_clean.columns:
        # Catch both true NaN values and "nan" text strings left over from casting
        df_lenders_clean[col] = df_lenders_clean[col].replace(["nan", "NaN", "", "None"], "Unknown")
        df_lenders_clean[col] = df_lenders_clean[col].fillna("Unknown")

# mpute numerical missing indices with a logical baseline count
if "loan_purchase_num" in df_lenders_clean.columns:
    df_lenders_clean["loan_purchase_num"] = df_lenders_clean["loan_purchase_num"].fillna(0.0)

# 3. Final verification print to ensure 0 nulls remain
print("=== POST-IMPUTATION DATA QUALITY REPORT ===")
print(df_lenders_clean.isnull().sum())
print(f"\nFinal Verified Shape: {df_lenders_clean.shape}")

=== POST-IMPUTATION DATA QUALITY REPORT ===
permanent_name       0
display_name         0
city                 0
state                0
country_code         0
member_since         0
occupation           0
loan_because         0
loan_purchase_num    0
invited_by           0
num_invited          0
dtype: int64

Final Verified Shape: (2349174, 11)


In [59]:
# Create a local reference copy for the export step
df_lenders_export = df_lenders_clean.copy()

# Save to your dedicated archive folder without writing the index row
df_lenders_export.to_csv(
    r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_lenders.csv",
    index=False
)

print("Lenders dataset clean state export successful!")

Lenders dataset clean state export successful!
